In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

# Préparation des données
Récupération et décompression de l'archive `ml-100k.zip` dans le répertoire de travail.

In [ ]:
import os, urllib.request, zipfile

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve("http://files.grouplens.org/datasets/movielens/ml-100k.zip", "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall(".")

## Data matrix

In [ ]:
import pandas
f = open("ml-100k/u.data",'r', encoding = 'latin_1')
rating_data = pandas.read_csv(f, delimiter='\t', header = None)
rating_data.columns=['user_id', 'item_id', 'rating', 'timestamp']
rating_data

In [ ]:
row = rating_data.user_id #- 1 # !! numerotation à partir de 0 en python
col = rating_data.item_id #- 1
rating = rating_data.rating

X = sp.coo_matrix((rating, (row , col))).toarray()

In [ ]:
plt.figure(figsize=(35,12))
plt.imshow(X, aspect='auto')

## Informations sur les films

In [ ]:
f = open("ml-100k/u.item",'r', encoding = 'latin_1')
film_data = pandas.read_csv(f, delimiter='|', header = None)
film_data.columns=['item_id', 'title', 'date', 'nan', 'url', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
film_data.set_index('item_id')
del film_data['nan']
film_data

**Exo : affichez le titre et le genre des dix premiers films de la liste.**

In [ ]:
genres = film_data.columns[4:]   # colonnes binaires de genre ('unknown' ... 'Western')

dix_films = film_data.head(10)
pandas.DataFrame({
    'title': dix_films['title'],
    'genre': dix_films[genres].apply(lambda r: ', '.join(genres[r == 1]), axis=1)
})

# Exercice 1 : calculs d'affinités

Remarque : les indices n'ont pas été décalés de 1 lors de la construction de `X` (ligne 0 et colonne 0 vides). Le film d'indice `idx` de `film_data` correspond donc à la colonne `film_data.item_id[idx]` (= `idx + 1`) de `X`.

In [ ]:
def simcos(x, y):
    d = np.sqrt(np.dot(x, x) * np.dot(y, y))
    return np.dot(x, y) / d if d > 0 else 0.0

def simcorr(x, y):
    xc, yc = x - np.mean(x), y - np.mean(y)
    d = np.sqrt(np.dot(xc, xc) * np.dot(yc, yc))
    return np.dot(xc, yc) / d if d > 0 else 0.0

def tableau_sim(M, sim, nb=5):
    films = film_data.item_id[:nb]
    titres = film_data.title[:nb]
    S = np.array([[sim(M[:, a], M[:, b]) for b in films] for a in films])
    return pandas.DataFrame(S, index=titres, columns=titres).round(3)

**Similarité cosinus entre les 5 premiers films**

In [ ]:
tableau_sim(X, simcos)

**Similarité de corrélation entre les 5 premiers films**

In [ ]:
tableau_sim(X, simcorr)

**Comparaison :** la similarité cosinus est calculée sur les notes brutes, où 0 représente une absence de note. Deux films sont jugés proches dès qu'ils sont notés par les mêmes utilisateurs : toutes les valeurs sont positives (entre 0,21 et 0,50). La corrélation retire d'abord la moyenne de chaque film : les valeurs sont plus faibles (entre 0,13 et 0,41) mais l'ordre des paires reste globalement le même (la paire la plus proche est GoldenEye / Get Shorty dans les deux cas). La moyenne d'une colonne, calculée sur tous les utilisateurs dont une majorité de 0, est faible : le centrage modifie peu les vecteurs.

**Les dix films les plus similaires à 'Citizen Kane (1941)'**

In [ ]:
idx_ck = film_data.index[film_data.title == 'Citizen Kane (1941)'][0]

def plus_similaires(M, idx, sim, n=10):
    j = film_data.item_id[idx]
    s = pandas.Series([sim(M[:, j], M[:, k]) for k in film_data.item_id], index=film_data.index)
    s = s.drop(idx).sort_values(ascending=False).head(n)   # on retire le film lui-même
    return pandas.DataFrame({'title': film_data.title[s.index], 'similarite': s.round(3)})

In [ ]:
plus_similaires(X, idx_ck, simcos)

In [ ]:
plus_similaires(X, idx_ck, simcorr)

**Conclusion :** les deux listes sont presque identiques (Casablanca, Vertigo, Chinatown, North by Northwest, 2001...) : ce sont des classiques du cinéma, cohérents avec Citizen Kane. Sur `X`, les absences de note (0) dominent : ces mesures traduisent surtout le fait que les films sont vus par les mêmes utilisateurs (ici, un public cinéphile), et non le fait qu'ils soient appréciés de la même façon. Une note de 1 rapproche deux films autant qu'une note de 5.

## Soustraction de la moyenne

In [ ]:
(m,n) = X.shape
X_norm = np.zeros((m,n))
for i in range(1, m):
    indices = np.where(X[i,:]>0)
    X_norm[i,indices] = X[i,indices] - np.mean(X[i,indices])

In [ ]:
plt.figure(figsize=(35,12))
plt.imshow(X_norm, aspect='auto')

# Exercice 1 bis
Mêmes questions avec `X_norm` au lieu de `X`.

In [ ]:
tableau_sim(X_norm, simcos)

In [ ]:
tableau_sim(X_norm, simcorr)

In [ ]:
plus_similaires(X_norm, idx_ck, simcos)

In [ ]:
plus_similaires(X_norm, idx_ck, simcorr)

**Conclusion :** après soustraction de la moyenne de chaque utilisateur, une note devient positive si l'utilisateur a préféré le film à sa moyenne et négative sinon, l'absence de note restant à 0. La similarité mesure alors l'accord des goûts et non plus seulement la co-notation. Pour les 5 premiers films, les similarités tombent près de 0 et certaines deviennent négatives (Get Shorty / Copycat : -0,15) : être vus par les mêmes personnes ne signifie pas être aimés par elles. Pour Citizen Kane, la liste reste proche (Casablanca, Vertigo, Chinatown en tête) avec des valeurs plus faibles (0,30 à 0,48), et de nouveaux classiques apparaissent (Rear Window, Sunset Blvd., The Godfather) : ces films sont non seulement vus mais aussi appréciés par les mêmes utilisateurs. Les colonnes de `X_norm` étant de moyenne proche de 0, cosinus et corrélation donnent quasiment les mêmes résultats.

# Exercice 2 : profiling

**Films notés par l'utilisateur 678, avec les notes associées**

In [ ]:
i = 678
films_678 = np.where(X[i, :] > 0)[0]          # numéros de colonnes = item_id
pandas.DataFrame({'item_id': films_678,
                  'title': film_data.title.values[films_678 - 1],
                  'note': X[i, films_678]})

**Fonction `calcule_voisins(i, K)`** : similarité cosinus entre les vecteurs lignes de `X`.

In [ ]:
def calcule_voisins(i, K):
    sims = np.array([simcos(X[i, :], X[u, :]) for u in range(m)])
    sims[i] = -np.inf                     # on exclut l'utilisateur lui-même
    return np.argsort(sims)[::-1][:K]     # K utilisateurs les plus similaires

In [ ]:
voisins = calcule_voisins(i,50)
print(voisins)

## Prédiction 1 : moyenne pondérée

In [ ]:
def moyenne_notes(u):
    return np.mean(X[u, X[u, :] > 0])     # moyenne sur les notes 1 à 5 uniquement

def moy_pond(i, voisins, j):
    m_i = moyenne_notes(i)
    v_j = [v for v in voisins if X[v, j] > 0]   # voisins ayant noté le film j
    if len(v_j) == 0:
        return m_i
    sims = np.array([simcos(X[i, :], X[v, :]) for v in v_j])
    ecarts = np.array([X[v, j] - moyenne_notes(v) for v in v_j])
    return m_i + np.sum(sims * ecarts) / np.sum(sims)

## Prédiction 2 : le vote

In [ ]:
def vote(voisins, j):
    notes = X[voisins, j]
    notes = notes[notes > 0].astype(int)       # voisins ayant noté le film j
    if len(notes) == 0:
        return np.nan                           # aucun voisin n'a noté le film
    return np.bincount(notes, minlength=6)[1:].argmax() + 1   # note la plus votée

# Exercice 3 : prédiction

**Note réelle et prédictions pour les films de l'utilisateur 678 (K = 50)**

In [ ]:
voisins = calcule_voisins(i, 50)
resultats = pandas.DataFrame({
    'title': film_data.title.values[films_678 - 1],
    'note reelle': X[i, films_678],
    'moy_pond': [round(moy_pond(i, voisins, j), 2) for j in films_678],
    'vote': [vote(voisins, j) for j in films_678]})
resultats

**Erreur moyenne en fonction du nombre de voisins**

Erreur absolue moyenne entre note réelle et prédiction. Pour comparer les deux méthodes sur les mêmes films, on ne garde que les films pour lesquels au moins un voisin a voté (sinon `vote` ne donne pas de prédiction).

In [ ]:
liste_K = [5, 10, 20, 30, 50, 75, 100, 150, 200, 300]
err_pond, err_vote = [], []

for K in liste_K:
    voisins = calcule_voisins(i, K)
    reel = X[i, films_678]
    p_pond = np.array([moy_pond(i, voisins, j) for j in films_678])
    p_vote = np.array([vote(voisins, j) for j in films_678])
    ok = ~np.isnan(p_vote)
    err_pond.append(np.mean(np.abs(p_pond[ok] - reel[ok])))
    err_vote.append(np.mean(np.abs(p_vote[ok] - reel[ok])))

print('Erreur moyenne sur tous les K : moy_pond = %.3f, vote = %.3f' % (np.mean(err_pond), np.mean(err_vote))))
pandas.DataFrame({'K': liste_K, 'erreur moy_pond': err_pond, 'erreur vote': err_vote}).round(3)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(liste_K, err_pond, 'o-', label='moyenne pondérée')
plt.plot(liste_K, err_vote, 's-', label='vote')
plt.xlabel('nombre de voisins K')
plt.ylabel('erreur absolue moyenne')
plt.legend()
plt.show()

**Conclusion :** sur les 28 films de l'utilisateur 678, aucune méthode ne l'emporte nettement : l'erreur moyenne sur les K testés est d'environ 0,86 pour la moyenne pondérée et 0,85 pour le vote. Le vote est très instable selon K (de 0,75 à 0,93) : une note entière, choisie à la majorité, tombe juste ou se trompe d'un point entier. La moyenne pondérée est plus régulière (de 0,83 à 0,90) et s'améliore légèrement quand K augmente. Elle tient compte de la sévérité de chaque utilisateur et du degré de similarité des voisins. Avec seulement 28 films, l'écart entre les deux méthodes n'est pas significatif ; la moyenne pondérée est la plus fiable car la moins sensible au choix de K.